# Task 4 — Pick the Right Chart, Five Times

**Dataset:** `mpg.csv` — 398 cars from 1970–1982  
Columns: `mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin`, `name`  
`horsepower` is missing for 6 cars.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('data/mpg.csv')
print(f'Shape: {df.shape}')
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nOrigins: {df["origin"].unique()}')
print(f'Model years: {sorted(df["model_year"].unique())}')

# Add full year column for readability
df['year'] = df['model_year'] + 1900
print(df.head(3))

## Q1 — Did cars get more fuel-efficient between 1970 and 1982?

In [ ]:
yearly_avg = df.groupby('year')['mpg'].agg(['mean', 'median', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(yearly_avg['year'],
                yearly_avg['mean'] - yearly_avg['std'],
                yearly_avg['mean'] + yearly_avg['std'],
                alpha=0.2, color='steelblue', label='±1 SD')
ax.plot(yearly_avg['year'], yearly_avg['mean'], color='steelblue',
        linewidth=2.5, marker='o', markersize=5, label='Mean MPG')

ax.set_title('Average Fuel Efficiency Jumped from 17 to 31 MPG Between 1970 and 1982',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('Fuel Efficiency (MPG)')
ax.set_ylim(0, 45)
ax.set_xticks(yearly_avg['year'])
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.tight_layout()
plt.savefig('task4_q1_mpg_trend.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart choice: Line chart.**  
This is a trend question over an ordered time axis — a line chart is the canonical choice. It shows the direction and pace of change directly. The shaded band adds honest uncertainty (±1 SD) without complicating the main message.

## Q2 — Do American, European and Japanese cars differ in fuel efficiency?

In [ ]:
origin_colors = {'usa': '#4E79A7', 'europe': '#E15759', 'japan': '#F28E2B'}
origin_labels = {'usa': 'USA', 'europe': 'Europe', 'japan': 'Japan'}
origins = ['usa', 'europe', 'japan']

fig, ax = plt.subplots(figsize=(8, 6))

data_by_origin = [df[df['origin'] == o]['mpg'].dropna().values for o in origins]
bp = ax.boxplot(data_by_origin,
                labels=[origin_labels[o] for o in origins],
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                flierprops=dict(marker='o', markersize=4, alpha=0.4))

for patch, origin in zip(bp['boxes'], origins):
    patch.set_facecolor(origin_colors[origin])
    patch.set_alpha(0.7)

# Add individual points
for i, (origin, data) in enumerate(zip(origins, data_by_origin), 1):
    jitter = np.random.RandomState(42).uniform(-0.15, 0.15, len(data))
    ax.scatter(i + jitter, data, color=origin_colors[origin],
               s=12, alpha=0.35, zorder=3)

ax.set_title('Japanese and European Cars Are More Fuel-Efficient than American Cars',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Country of Origin')
ax.set_ylabel('Fuel Efficiency (MPG)')
ax.set_ylim(0, 50)

plt.tight_layout()
plt.savefig('task4_q2_mpg_origin.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart choice: Box plot with jittered points.**  
This is a distribution-comparison question across three categories. A box plot shows median, IQR, and outliers for each group in one compact glyph. I overlay jittered raw points to show the actual data density — a box plot alone hides sample size and within-box shape. A bar chart showing only means would lose all distributional information.

## Q3 — What is the relationship between a car's weight and its fuel efficiency?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(df['weight'], df['mpg'], color='steelblue', s=18, alpha=0.45, edgecolors='none')

# Fit and plot trend line
m, b = np.polyfit(df['weight'], df['mpg'], 1)
x_range = np.linspace(df['weight'].min(), df['weight'].max(), 200)
ax.plot(x_range, m * x_range + b, color='#E15759', linewidth=2, label=f'Trend (slope={m:.4f})')

ax.set_title('Heavier Cars Get Fewer Miles per Gallon — a Strong Negative Relationship',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Vehicle Weight (lbs)')
ax.set_ylabel('Fuel Efficiency (MPG)')
ax.set_ylim(0, 50)
ax.set_xlim(0)
ax.legend()

# Print correlation
r = df[['weight', 'mpg']].corr().iloc[0, 1]
ax.text(0.97, 0.95, f'r = {r:.2f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('task4_q3_weight_mpg.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart choice: Scatter plot.**  
A relationship question between two continuous variables calls for a scatter plot. It shows every individual car as one point, revealing the direction (negative), strength (strong, r ≈ −0.83), and shape (roughly linear) of the relationship. No other chart type can show this combination.

## Q4 — How is the number of cylinders distributed across the fleet?

In [ ]:
cyl_counts = df['cylinders'].value_counts().sort_index()
print(cyl_counts)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(cyl_counts.index.astype(str), cyl_counts.values,
              color='steelblue', edgecolor='white', width=0.6)

# Label each bar
for bar, val in zip(bars, cyl_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontsize=10)

ax.set_title('Four- and Eight-Cylinder Cars Dominate; Three- and Five-Cylinder Are Rare',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Number of Cylinders')
ax.set_ylabel('Count of Cars')
ax.set_ylim(0, max(cyl_counts.values) * 1.15)

plt.tight_layout()
plt.savefig('task4_q4_cylinders.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart choice: Bar chart.**  
Number of cylinders is a discrete categorical variable (3, 4, 5, 6, 8). A bar chart is the right tool for comparing counts across distinct categories. A histogram would be appropriate only if cylinders were truly continuous (which it is not — there are no 4.5-cylinder cars). Each bar's height is a count, so bars correctly start at zero.

## Q5 — Is the improvement in MPG explained by cars getting lighter?

In [ ]:
yearly_weight = df.groupby('year')['weight'].mean().reset_index()
yearly_mpg = df.groupby('year')['mpg'].mean().reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))

# NOTE: The lecture says "never two y-axes". I am using a scatter approach instead.
# Showing weight vs mpg across years as a connected scatter plot.
ax1.set_visible(False)

fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(yearly_weight['weight'], yearly_mpg['mpg'],
                     c=yearly_weight['year'], cmap='viridis', s=80, zorder=3)

# Connect points chronologically
ax.plot(yearly_weight['weight'], yearly_mpg['mpg'], color='gray',
        linewidth=1, alpha=0.5, zorder=2)

# Label first and last year
for _, row in yearly_weight.iterrows():
    yr = int(row['year'])
    mpg_val = yearly_mpg[yearly_mpg['year'] == yr]['mpg'].values[0]
    if yr in [1970, 1982, 1974, 1978]:
        ax.annotate(str(yr), xy=(row['weight'], mpg_val),
                    xytext=(5, 4), textcoords='offset points', fontsize=8)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Model Year')

ax.set_title('Cars Got Both Lighter AND More Efficient — Weight Loss Partly Explains the MPG Gain\n(each point = fleet average for that year; arrow of time goes up-left)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Average Vehicle Weight (lbs)')
ax.set_ylabel('Average Fuel Efficiency (MPG)')

plt.tight_layout()
plt.savefig('task4_q5_weight_mpg_years.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Weight correlation with year: {df[["year","weight"]].corr().iloc[0,1]:.2f}')

**Chart choice: Connected scatter plot (path chart).**  
This question asks about the relationship between three variables simultaneously: MPG, weight, and time. A connected scatter plot encodes two quantitative variables on the axes (weight and MPG) and uses a colour gradient plus the connecting line to encode the time dimension. This avoids a dual y-axis, which would be forbidden by the lecture rules and genuinely harder to read. The path shows that as years pass (darker colour → higher year), both weight falls and MPG rises — confirming the causal story.

## Redraw — Weakest Chart (Q4) Improved

In [ ]:
# Q4 was weakest because it showed only counts, not the MPG implication.
# Redraw: show both count AND mean MPG, sorted by cylinder count.

cyl_summary = df.groupby('cylinders').agg(
    count=('mpg', 'count'),
    mean_mpg=('mpg', 'mean')
).reset_index().sort_values('cylinders')

fig, ax = plt.subplots(figsize=(9, 5))

# Colour bars by mean MPG using a sequential palette
norm = plt.Normalize(cyl_summary['mean_mpg'].min(), cyl_summary['mean_mpg'].max())
cmap = plt.cm.RdYlGn
colors = [cmap(norm(v)) for v in cyl_summary['mean_mpg']]

bars = ax.bar(cyl_summary['cylinders'].astype(str), cyl_summary['count'],
              color=colors, edgecolor='white', width=0.6)

# Label bars with count and MPG
for bar, row in zip(bars, cyl_summary.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            f'n={row.count}\n{row.mean_mpg:.1f} mpg',
            ha='center', va='bottom', fontsize=8)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6)
cbar.set_label('Mean MPG (green = more efficient)')

ax.set_title('4-Cylinder Cars Are Most Common and Most Efficient; 8-Cylinders Are Second Most Common but Least Efficient',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Number of Cylinders')
ax.set_ylabel('Count of Cars')
ax.set_ylim(0, 230)

plt.tight_layout()
plt.savefig('task4_q4_cylinders_improved.png', dpi=150, bbox_inches='tight')
plt.show()

**What I changed:** The original Q4 bar chart showed counts only, which answered the distribution question but nothing else. The redraw uses bar colour to encode mean MPG on a green-to-red sequential scale, so the reader simultaneously sees how common each cylinder count is (bar height) and how efficient those cars typically are (colour). The annotations add exact counts and mean MPG values. This doubles the information density without adding a second y-axis.